# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their field and column `@id`s.

We first list all record sets and their content by `@id`. This helps in selecting data entities for extraction and processing. 

In [ ]:
# List available record sets and fields (@id)
from pprint import pprint

def get_record_sets(ds):
    # Get all record sets in the schema
    # Returns list of record set objects
    rs = []
    if hasattr(ds.metadata, 'record_sets'):
        rs = ds.metadata.record_sets
    # In some schemas, record_sets may not be populated, try via dataset.find()
    if not rs:
        # Use dataset.find to locate all entities of type RecordSet
        rs = ds.find('@type', 'RecordSet')
    return rs

record_sets = get_record_sets(dataset)

if not record_sets:
    print("No record sets found in this Croissant package.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        # List the fields and columns in the record set
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields and columns @id:")
        for field in fields:
            fid = field if isinstance(field, str) else field.get('@id', field)
            print(f"    - {fid}")
        print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

**All references to record sets, fields, and columns are identified by their `@id`.**

The code block below extracts data from each available record set (by its `@id`) and loads it into a `pandas` DataFrame.

In [ ]:
from mlcroissant._src.structure.dataset import CroissantNotFoundError

record_sets = get_record_sets(dataset)
# List of record set @id's
record_set_ids = [r['@id'] for r in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
        else:
            print(f"  ...no records returned.")
    except CroissantNotFoundError:
        print(f"  ...record set {record_set_id} could not be loaded by mlcroissant.")

if dataframes:
    example_rs_id = next(iter(dataframes.keys()))
    print(f"\nColumns in '{example_rs_id}':")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())
else:
    print("No tabular dataframes constructed.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping or aggregating data.

In this section, we select a numeric field (by `@id`), filter, normalize, and optionally group the data.

In [ ]:
# Select a record set to analyze
if not dataframes:
    print("No data available for EDA.")
else:
    # For demonstration, pick the first available record set
    record_set_id = example_rs_id
    df = dataframes[record_set_id]

    print(f"Performing EDA on record set: {record_set_id}")

    # Try to auto-select a likely numeric field by dtype
    numeric_columns = df.select_dtypes(include=['number']).columns
    if len(numeric_columns) == 0:
        print("No numeric fields found for filtering and normalization.")
    else:
        numeric_field = numeric_columns[0]  # use the first numeric field
        print(f"Using numeric field for demonstration: {numeric_field}")

        # Filter with threshold
        threshold = df[numeric_field].mean()  # use mean as demonstration threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by a categorical field, if available
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        # Exclude obvious non-group keys
        possible_group = [c for c in cat_cols if c.lower() not in ['@id', 'id', '']]
        if possible_group:
            group_field = possible_group[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data available to visualize.")
else:
    # We'll reuse filtered_df and numeric_field from the previous cell, if present
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group_field if available
    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(9,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook introduced programmatic access to the FAIR² dataset schema and records using [`mlcroissant`](https://pypi.org/project/mlcroissant/). We demonstrated how to:

- Access the dataset's Croissant metadata and enumerate all available record sets by their `@id`.
- Load records from specific record sets for downstream tabular analysis.
- Filter, normalize, group, and visualize data, always referencing fields by their unique `@id`.

This approach supports reproducible, interoperable, and modular data workflows—an important step toward FAIR+AI best practices.